In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv("data/raw/online_retail_II.csv")

print("Shape (rows, columns):", df.shape)
display(df.head())
print(df.dtypes)

Shape (rows, columns): (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [3]:
print("Missing values per column:")
print(df.isna().sum())

print("\nDistinct invoices:", df["Invoice"].nunique())
print("Distinct products (StockCode):", df["StockCode"].nunique())
print("Distinct customers:", df["Customer ID"].nunique())
print("Distinct countries:", df["Country"].nunique())

Missing values per column:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Distinct invoices: 53628
Distinct products (StockCode): 5305
Distinct customers: 5942
Distinct countries: 43


In [4]:
lines_per_invoice = df.groupby("Invoice").size()
print("Lines per invoice:")
print(lines_per_invoice.describe())

print("\nRows with Quantity < 0:", (df["Quantity"] < 0).sum())
print("Rows with Price <= 0:  ", (df["Price"] <= 0).sum())
print("Invoices starting with C:", df["Invoice"].str.startswith("C").sum())

Lines per invoice:
count    53628.000000
mean        19.903241
std         41.252573
min          1.000000
25%          2.000000
50%          9.000000
75%         23.000000
max       1350.000000
dtype: float64

Rows with Quantity < 0: 22950
Rows with Price <= 0:   6207
Invoices starting with C: 19494


In [5]:
is_cancel = df["Invoice"].str.startswith("C")
is_negqty = df["Quantity"] < 0

print(pd.crosstab(is_cancel, is_negqty,
                  rownames=["Starts with C"], colnames=["Quantity < 0"]))

print("Price < 0: ", (df["Price"] < 0).sum())
print("Price == 0:", (df["Price"] == 0).sum())

print("\nNegative prices:")
print(df[df["Price"] < 0])

print("\nSample of zero prices:")
print(df[df["Price"] == 0].head(10))

Quantity < 0     False  True 
Starts with C                
False          1044420   3457
True                 1  19493
Price < 0:  5
Price == 0: 6202

Negative prices:
        Invoice StockCode      Description  Quantity          InvoiceDate  \
179403  A506401         B  Adjust bad debt         1  2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1  2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1  2010-10-20 12:04:00   
825444  A563186         B  Adjust bad debt         1  2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1  2011-08-12 14:52:00   

           Price  Customer ID         Country  
179403 -53594.36          NaN  United Kingdom  
276274 -44031.79          NaN  United Kingdom  
403472 -38925.87          NaN  United Kingdom  
825444 -11062.06          NaN  United Kingdom  
825445 -11062.06          NaN  United Kingdom  

Sample of zero prices:
     Invoice StockCode   Description  Quantity          

In [6]:
is_cancel = df["Invoice"].str.startswith("C")
neg_not_c = df[(df["Quantity"] < 0) & (~is_cancel)]

print("Rows:", len(neg_not_c))
print("With Price == 0:", (neg_not_c["Price"] == 0).sum())
print("With no Customer ID:", neg_not_c["Customer ID"].isna().sum())
print()
print(neg_not_c["Description"].value_counts(dropna=False).head(10))

print(df[is_cancel & (df["Quantity"] >= 0)])

first_char = df["Invoice"].str[0]
print(first_char[~first_char.str.isdigit()].value_counts())

Rows: 3457
With Price == 0: 3457
With no Customer ID: 3457

Description
NaN                      2689
check                     123
damages                    84
?                          83
damaged                    78
missing                    27
sold as set on dotcom      20
Damaged                    17
smashed                     9
thrown away                 9
Name: count, dtype: int64
       Invoice StockCode Description  Quantity          InvoiceDate   Price  \
76799  C496350         M      Manual         1  2010-02-01 08:24:00  373.57   

       Customer ID         Country  
76799          NaN  United Kingdom  
Invoice
C    19494
A        6
Name: count, dtype: int64


In [20]:
pattern = r"^\d{5}[A-Za-z]*$"
odd = df[~df["StockCode"].str.match(pattern)]

print("Rows with a non-product-looking StockCode:", len(odd))
print(odd["StockCode"].value_counts().head(15))

Rows with a non-product-looking StockCode: 6094
StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
Name: count, dtype: int64


In [8]:
free = df[(df["Price"] == 0) & (df["Quantity"] > 0)]
print("Rows:", len(free))
print("With no Customer ID:", free["Customer ID"].isna().sum())
print(free["Description"].value_counts(dropna=False).head(8))

Rows: 2745
With no Customer ID: 2674
Description
NaN                             1693
check                             39
found                             28
OWL DOORSTOP                      15
adjustment                        14
POLYESTER FILLER PAD 45x45cm      12
amazon                            11
POLYESTER FILLER PAD 40x40cm      10
Name: count, dtype: int64


In [9]:
print(df[df["Invoice"].str.startswith("A")])

        Invoice StockCode      Description  Quantity          InvoiceDate  \
179403  A506401         B  Adjust bad debt         1  2010-04-29 13:36:00   
276274  A516228         B  Adjust bad debt         1  2010-07-19 11:24:00   
403472  A528059         B  Adjust bad debt         1  2010-10-20 12:04:00   
825443  A563185         B  Adjust bad debt         1  2011-08-12 14:50:00   
825444  A563186         B  Adjust bad debt         1  2011-08-12 14:51:00   
825445  A563187         B  Adjust bad debt         1  2011-08-12 14:52:00   

           Price  Customer ID         Country  
179403 -53594.36          NaN  United Kingdom  
276274 -44031.79          NaN  United Kingdom  
403472 -38925.87          NaN  United Kingdom  
825443  11062.06          NaN  United Kingdom  
825444 -11062.06          NaN  United Kingdom  
825445 -11062.06          NaN  United Kingdom  


In [10]:
top_codes = odd["StockCode"].value_counts().head(15).index
for code in top_codes:
    names = odd.loc[odd["StockCode"] == code, "Description"].dropna().unique()[:2]
    print(code, "->", list(names))

free_cust = free[free["Customer ID"].notna()]
print("Rows:", len(free_cust))
print(free_cust.head(10))

dups = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print("Rows involved in duplicates (all copies):", len(dups))
print(dups.head(10))

POST -> ['POSTAGE']
DOT -> ['DOTCOM POSTAGE']
M -> ['Manual']
C2 -> ['CARRIAGE']
D -> ['Discount']
S -> ['SAMPLES']
BANK CHARGES -> [' Bank Charges', 'Bank Charges']
ADJUST -> ['Adjustment by john on 26/01/2010 16', 'Adjustment by john on 26/01/2010 17']
AMAZONFEE -> ['AMAZON FEE']
DCGS0058 -> ['MISO PRETTY  GUM']
gift_0001_20 -> ['Dotcomgiftshop Gift Voucher £20.00', 'to push order througha s stock was ']
gift_0001_30 -> ['Dotcomgiftshop Gift Voucher £30.00']
DCGSSGIRL -> ['update', 'GIRLS PARTY BAG']
DCGSSBOY -> ['update', 'BOYS PARTY BAG']
PADS -> ['PADS TO MATCH ALL CUSHIONS']
Rows: 71
      Invoice StockCode                      Description  Quantity  \
4674   489825     22076               6 RIBBONS EMPIRE          12   
6781   489998     48185              DOOR MAT FAIRY CAKE         2   
16107  490727         M                           Manual         1   
18738  490961     22065   CHRISTMAS PUDDING TRINKET POT          1   
18739  490961     22142     CHRISTMAS CRAFT WHITE FAI

In [3]:
dup_mask = df.duplicated(keep="first")
print("Duplicate rows (extra copies):", dup_mask.sum())
print("Invoices containing a duplicate:", df.loc[dup_mask, "Invoice"].nunique())

print("\nInvoices with the most duplicate rows:")
print(df.loc[dup_mask].groupby("Invoice").size().sort_values(ascending=False).head(10))

line_value = df["Quantity"] * df["Price"]
print("\nValue of the extra copies:", round(line_value[dup_mask].sum(), 2))
print("Value of all rows:        ", round(line_value.sum(), 2))

Duplicate rows (extra copies): 34335
Invoices containing a duplicate: 5391

Invoices with the most duplicate rows:
Invoice
537434    675
538071    652
537638    601
537237    597
536876    593
536592    592
537823    591
537240    568
537666    536
536544    527
dtype: int64

Value of the extra copies: 431716.87
Value of all rows:         19287250.57


In [4]:
dup_per_invoice = df.loc[dup_mask].groupby("Invoice").size()
lines = df.groupby("Invoice").size()

check = pd.DataFrame({
    "dup_rows": dup_per_invoice,
    "total_lines": lines[dup_per_invoice.index],
})
check["dup_share"] = (check["dup_rows"] / check["total_lines"]).round(2)

print(check["dup_rows"].describe())
print()
print(check.sort_values("dup_rows", ascending=False).head(10))

dates = pd.to_datetime(df.loc[dup_mask, "InvoiceDate"])
print("\nDuplicates date range:", dates.min(), "to", dates.max())
print(dates.dt.to_period("M").value_counts().sort_index())

count    5391.000000
mean        6.368948
std        27.885624
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max       675.000000
Name: dup_rows, dtype: float64

         dup_rows  total_lines  dup_share
Invoice                                  
537434        675         1350        0.5
538071        652         1304        0.5
537638        601         1202        0.5
537237        597         1194        0.5
536876        593         1186        0.5
536592        592         1184        0.5
537823        591         1182        0.5
537240        568         1136        0.5
537666        536         1072        0.5
536544        527         1054        0.5

Duplicates date range: 2009-12-01 11:34:00 to 2011-12-09 11:34:00
InvoiceDate
2009-12      506
2010-01      321
2010-02      330
2010-03      526
2010-04      410
2010-05      415
2010-06      514
2010-07      391
2010-08      367
2010-09      484
2010-10      849
2010-11     1431
2010-12    23

In [5]:
dup_per_invoice = df.loc[dup_mask].groupby("Invoice").size()
lines = df.groupby("Invoice").size()
share = dup_per_invoice / lines[dup_per_invoice.index]

print("Invoices with duplicates:", len(share))
print("Of those, share exactly 0.5:", (share == 0.5).sum())
print("Duplicate rows in those invoices:", dup_per_invoice[share == 0.5].sum())

Invoices with duplicates: 5391
Of those, share exactly 0.5: 1015
Duplicate rows in those invoices: 18236


In [6]:
same_half = share[share == 0.5].index
sizes = lines[same_half]

print(sizes.describe())
print("Invoices in this group with 20+ lines:", (sizes >= 20).sum())
print("Invoices in this group with fewer than 6 lines:", (sizes < 6).sum())

count    1015.000000
mean       35.933005
std       120.376413
min         2.000000
25%         2.000000
50%        12.000000
75%        32.000000
max      1350.000000
dtype: float64
Invoices in this group with 20+ lines: 433
Invoices in this group with fewer than 6 lines: 368


In [7]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(df["InvoiceDate"].dtype)
print("First:", df["InvoiceDate"].min())
print("Last: ", df["InvoiceDate"].max())

datetime64[ns]
First: 2009-12-01 07:45:00
Last:  2011-12-09 12:50:00


In [8]:
big_double = share[(share == 0.5) & (lines[share.index] >= 20)].index
invoice_dates = df.groupby("Invoice")["InvoiceDate"].min()[big_double]

print("Invoices in this group:", len(big_double))
print("Earliest:", invoice_dates.min())
print("Latest:  ", invoice_dates.max())
print()
print(invoice_dates.dt.to_period("M").value_counts().sort_index())

Invoices in this group: 433
Earliest: 2010-12-01 08:34:00
Latest:   2011-10-26 12:46:00

InvoiceDate
2010-12    432
2011-10      1
Freq: M, Name: count, dtype: int64


In [9]:
cols = ["Invoice", "StockCode", "Description", "Quantity",
        "InvoiceDate", "Price", "Customer ID", "Country"]

half = share[share == 0.5].index
sizes = lines[half]

groups = {
    "20+ lines":     sizes[sizes >= 20].index,
    "6 to 19 lines": sizes[(sizes >= 6) & (sizes < 20)].index,
    "under 6 lines": sizes[sizes < 6].index,
}

for name, invoices in groups.items():
    sub = df[df["Invoice"].isin(invoices)]
    copies = sub.groupby(cols, dropna=False).size()
    lo = copies.groupby(level="Invoice").min()
    hi = copies.groupby(level="Invoice").max()
    clean = ((lo == 2) & (hi == 2)).sum()
    print(name, "| invoices:", len(invoices), "| clean doubling:", clean)

20+ lines | invoices: 433 | clean doubling: 432
6 to 19 lines | invoices: 214 | clean doubling: 214
under 6 lines | invoices: 368 | clean doubling: 368


In [10]:
mid = groups["6 to 19 lines"]
mid_dates = df.groupby("Invoice")["InvoiceDate"].min()[mid]
print(mid_dates.dt.to_period("M").value_counts().sort_index())

InvoiceDate
2010-12    214
Freq: M, Name: count, dtype: int64


In [11]:
small = groups["under 6 lines"]
d = df.groupby("Invoice")["InvoiceDate"].min()[small]

print("Invoices:", len(d))
print("Created in Dec 2010:", ((d >= "2010-12-01") & (d < "2011-01-01")).sum())
print()
print(d.dt.to_period("M").value_counts().sort_index())

Invoices: 368
Created in Dec 2010: 358

InvoiceDate
2010-05      2
2010-09      1
2010-11      1
2010-12    358
2011-04      1
2011-06      2
2011-07      1
2011-09      1
2011-12      1
Freq: M, Name: count, dtype: int64


In [12]:
first_date = df.groupby("Invoice")["InvoiceDate"].min()
dec = first_date[(first_date >= "2010-12-01") & (first_date < "2011-01-01")].index

dup_count = df.loc[dup_mask].groupby("Invoice").size()
dec_with_dup = dup_count.index.intersection(dec)

print("Invoices created in Dec 2010:", len(dec))
print("Of those, with at least one duplicate row:", len(dec_with_dup))
print("Of those, share exactly 0.5:", (share[dec_with_dup] == 0.5).sum())
print("Duplicate rows in Dec 2010 invoices:", dup_count[dec_with_dup].sum())

Invoices created in Dec 2010: 2025
Of those, with at least one duplicate row: 1156
Of those, share exactly 0.5: 1004
Duplicate rows in Dec 2010 invoices: 23023


In [13]:
doubled = dec_with_dup[share[dec_with_dup] == 0.5]

table = pd.DataFrame({
    "invoices": first_date[dec].dt.date.value_counts(),
    "doubled": first_date[doubled].dt.date.value_counts(),
}).fillna(0).astype(int).sort_index()

table["doubled_share"] = (table["doubled"] / table["invoices"]).round(2)
print(table)

             invoices  doubled  doubled_share
InvoiceDate                                  
2010-12-01        143      130           0.91
2010-12-02        167      161           0.96
2010-12-03        108      104           0.96
2010-12-05         95       67           0.71
2010-12-06        133      119           0.89
2010-12-07        111      105           0.95
2010-12-08        148      142           0.96
2010-12-09        183      176           0.96
2010-12-10         91        0           0.00
2010-12-12         51        0           0.00
2010-12-13         84        0           0.00
2010-12-14        144        0           0.00
2010-12-15         98        0           0.00
2010-12-16        150        0           0.00
2010-12-17         72        0           0.00
2010-12-19         25        0           0.00
2010-12-20         96        0           0.00
2010-12-21         66        0           0.00
2010-12-22         22        0           0.00
2010-12-23         38        0    

In [14]:
in_window = first_date[(first_date >= "2010-12-01") & (first_date < "2010-12-10")].index
after = first_date[(first_date >= "2010-12-10") & (first_date < "2011-01-01")].index
has_dup = dup_count.index

print("Invoices Dec 1-9:", len(in_window))
print("  doubled (share 0.5):", len(in_window.intersection(doubled)))
print("  duplicates but not 0.5:", len(in_window.intersection(has_dup).difference(doubled)))
print("  no duplicate:", len(in_window.difference(has_dup)))
print()
print("Invoices Dec 10-31:", len(after))
print("  with any duplicate:", len(after.intersection(has_dup)))

Invoices Dec 1-9: 1088
  doubled (share 0.5): 1004
  duplicates but not 0.5: 84
  no duplicate: 0

Invoices Dec 10-31: 937
  with any duplicate: 68


In [15]:
w84 = in_window.intersection(has_dup).difference(doubled)

print("Invoices:", len(w84))
print()
print("Share of duplicate lines:")
print(share[w84].describe())
print()
print("Total lines per invoice:")
print(lines[w84].describe())

Invoices: 84

Share of duplicate lines:
count    84.000000
mean      0.534518
std       0.020391
min       0.504237
25%       0.517241
50%       0.530777
75%       0.548214
max       0.602041
dtype: float64

Total lines per invoice:
count     84.000000
mean     102.761905
std       57.833083
min       18.000000
25%       64.000000
50%      100.000000
75%      128.500000
max      338.000000
dtype: float64


In [16]:
sub = df[df["Invoice"].isin(w84)]
copies = sub.groupby(cols, dropna=False).size()

print("Copy counts per distinct line:")
print(copies.value_counts().sort_index())

all_even = (copies % 2 == 0).groupby(level="Invoice").all()
print("\nInvoices where every line appears an even number of times:",
      all_even.sum(), "of", len(all_even))

Copy counts per distinct line:
2     3699
4      273
6       22
10       1
Name: count, dtype: int64

Invoices where every line appears an even number of times: 84 of 84


## Decision: handling duplicate rows

**What we found**
- 34,335 rows are exact copies of an earlier row, spread across 5,391 invoices.
- All 1,088 invoices created between 1st and 9th December 2010 contain duplicates. From 10 to 31 December, only 68 of 937 do (about 7%).
- In that window, 1,004 invoices have every distinct line exactly twice. The other 84 have every line an even number of times (2, 4, 6 or 10), which fits a doubled invoice that also contained genuine repeated lines.
- This is consistent with a one-off recording or export error. The data cannot show the cause.

**What we decide**
- No rows are deleted. We add two True/False columns:
  - `is_duplicate`: the row is a repeat of an earlier identical row.
  - `likely_double_entry`: the row is in the second half of a group of identical rows, on an invoice created 1-9 December 2010.
- Results are reported in two versions: with all rows, and without `likely_double_entry` rows.

**Limitations**
- The window comes from a daily table. The exact start and end are not known.
- Duplicates outside the window (including 11 invoices with a doubled pattern and 68 invoices in 10-31 December) are flagged as duplicates but not as likely double entries. They may be genuine repeated lines.
- That genuine repeats exist inside doubled invoices is an inference from even copy counts, not a proven fact.

In [17]:
df["is_duplicate"] = df.duplicated(keep="first")

event_rows = df["Invoice"].isin(in_window)
group_size = df.groupby(cols, dropna=False)["Invoice"].transform("size")
position = df.groupby(cols, dropna=False).cumcount()

df["likely_double_entry"] = event_rows & (position >= group_size / 2)

print("is_duplicate:", df["is_duplicate"].sum())
print("Rows in event invoices:", event_rows.sum())
print("likely_double_entry:", df["likely_double_entry"].sum())

is_duplicate: 34335
Rows in event invoices: 45046
likely_double_entry: 22523


In [18]:
line_value = df["Quantity"] * df["Price"]
flagged = df["likely_double_entry"]

print("Rows flagged:", flagged.sum())
print("Value of flagged rows:", round(line_value[flagged].sum(), 2))
print("Value of all rows:    ", round(line_value.sum(), 2))
print("Flagged share of value (%):",
      round(line_value[flagged].sum() / line_value.sum() * 100, 2))

print("\nFlagged rows outside Dec 1-9:", (flagged & ~event_rows).sum())

Rows flagged: 22523
Value of flagged rows: 377488.45
Value of all rows:     19287250.57
Flagged share of value (%): 1.96

Flagged rows outside Dec 1-9: 0


## classify the rows types 

In [21]:
all_odd = odd["StockCode"].value_counts()
print("Distinct odd codes:", len(all_odd))
print("Rows outside the top 15:", all_odd.iloc[15:].sum())
print()
print(all_odd.iloc[15:])

Distinct odd codes: 63
Rows outside the top 15: 174

StockCode
CRUK            16
gift_0001_10    16
TEST001         15
DCGS0076        15
DCGS0003        14
gift_0001_50     8
gift_0001_40     7
DCGS0069         6
B                6
m                5
DCGS0004         5
gift_0001_80     4
DCGS0066N        4
DCGS0072         4
gift_0001_70     3
DCGS0068         3
SP1002           3
ADJUST2          3
DCGS0070         3
DCGS0037         2
gift_0001_60     2
gift_0001_90     2
TEST002          2
DCGS0062         2
DCGS0027         1
DCGS0016         1
DCGS0006         1
DCGS0044         1
DCGSLBOY         1
GIFT             1
DCGS0059         1
DCGS0036         1
DCGSLGIRL        1
DCGS0053         1
DCGS0060         1
DCGS0039         1
DCGS0056         1
DCGS0041         1
C3               1
DCGS0075         1
47503J           1
DCGS0055         1
DCGS0057         1
DCGS0074         1
DCGS0073         1
DCGS0071         1
DCGS0066P        1
DCGS0067         1
Name: count, dtype: int64

In [22]:
rest = all_odd.iloc[15:].index
for code in rest:
    names = odd.loc[odd["StockCode"] == code, "Description"].dropna().unique()[:2]
    print(code, all_odd[code], list(names))

CRUK 16 ['CRUK Commission']
gift_0001_10 16 ['Dotcomgiftshop Gift Voucher £10.00']
TEST001 15 ['This is a test product.']
DCGS0076 15 ['SUNJAR LED NIGHT NIGHT LIGHT']
DCGS0003 14 ['BOXED GLASS ASHTRAY', 'ebay']
gift_0001_50 8 ['Dotcomgiftshop Gift Voucher £50.00']
gift_0001_40 7 ['Dotcomgiftshop Gift Voucher £40.00']
DCGS0069 6 ['OOH LA LA DOGS COLLAR', 'ebay']
B 6 ['Adjust bad debt']
m 5 ['Manual']
DCGS0004 5 ['HAYNES CAMPER SHOULDER BAG']
gift_0001_80 4 ['Dotcomgiftshop Gift Voucher £80.00']
DCGS0066N 4 ['NAVY CUDDLES DOG HOODIE']
DCGS0072 4 ['CAT CAMOUFLAGUE COLLAR']
gift_0001_70 3 ['Dotcomgiftshop Gift Voucher £70.00']
DCGS0068 3 ['DOGS NIGHT COLLAR', 'ebay']
SP1002 3 ["KID'S CHALKBOARD/EASEL"]
ADJUST2 3 ['Adjustment by Peter on Jun 25 2010 ']
DCGS0070 3 ['CAMOUFLAGE DOG COLLAR']
DCGS0037 2 ['KEY-RING CORKSCREW']
gift_0001_60 2 []
gift_0001_90 2 []
TEST002 2 ['This is a test product.']
DCGS0062 2 ['ROAD-RAGE CAR FRESHENER']
DCGS0027 1 []
DCGS0016 1 []
DCGS0006 1 []
DCGS0044 1 ['HAN

In [23]:
code = df["StockCode"].str.strip().str.upper()
shape_ok = code.str.match(r"^\d{5}[A-Z]*$")

desc = df["Description"]
not_upper = desc.notna() & (desc != desc.str.upper())

check = df[shape_ok & not_upper]
print("Rows:", len(check))
print("Distinct descriptions:", check["Description"].nunique())
print(check["Description"].str.strip().value_counts().head(40).to_string())

Rows: 4068
Distinct descriptions: 291
Description
BAG 250g SWIRLY MARBLES                598
BAG 125g SWIRLY MARBLES                596
POLYESTER FILLER PAD 40x40cm           398
POLYESTER FILLER PAD 45x45cm           372
3 TRADITIONAl BISCUIT CUTTERS  SET     268
BAG 500g SWIRLY MARBLES                240
FRENCH BLUE METAL DOOR SIGN No         186
check                                  162
damages                                 84
damaged                                 81
Next Day Carriage                       80
ESSENTIAL BALM 3.5g TIN IN ENVELOPE     80
POLYESTER FILLER PAD 45x30cm            65
POLYESTER FILLER PAD 30CMx30CM          52
POLYESTER FILLER PAD 65CMx65CM          52
FRENCH BLUE METAL DOOR SIGN, No         48
NUMBER TILE COTTAGE GARDEN No           42
NUMBER TILE VINTAGE FONT No             39
FOLK ART GREETING CARD,pack/12          31
found                                   28
missing                                 27
sold as set on dotcom                   20
Dama

In [24]:
check = df[shape_ok & not_upper]
zero = check["Price"] == 0

print("Rows:", len(check))
print("Zero price:", zero.sum(), "| Price > 0:", (~zero).sum())
print("Zero price AND no Customer ID:", (zero & check["Customer ID"].isna()).sum())
print()
print("Descriptions with Price > 0 (top 40):")
print(check.loc[~zero, "Description"].str.strip().value_counts().head(40).to_string())

Rows: 4068
Zero price: 875 | Price > 0: 3193
Zero price AND no Customer ID: 874

Descriptions with Price > 0 (top 40):
Description
BAG 250g SWIRLY MARBLES                598
BAG 125g SWIRLY MARBLES                596
POLYESTER FILLER PAD 40x40cm           388
POLYESTER FILLER PAD 45x45cm           360
3 TRADITIONAl BISCUIT CUTTERS  SET     268
BAG 500g SWIRLY MARBLES                240
FRENCH BLUE METAL DOOR SIGN No         178
ESSENTIAL BALM 3.5g TIN IN ENVELOPE     80
Next Day Carriage                       80
POLYESTER FILLER PAD 45x30cm            64
POLYESTER FILLER PAD 65CMx65CM          51
POLYESTER FILLER PAD 30CMx30CM          51
FRENCH BLUE METAL DOOR SIGN, No         48
NUMBER TILE COTTAGE GARDEN No           42
NUMBER TILE VINTAGE FONT No             39
FOLK ART GREETING CARD,pack/12          31
THE KING GIFT BAG 25x24x12cm            13
Dad's Cab Electronic Meter              11
Hypochondriac Pill Stress Ball           9
POLYESTER FILLER PAD 60x40cm             6
Dr. Jam's

In [25]:
words = (r"\b(?:carriage|postage|voucher|fee|commission|discount|manual|"
         r"adjust\w*|test|sample\w*|bad debt|dotcom|amazon|ebay|damag\w*|"
         r"check\w*|found|missing|lost|destroy\w*)\b")

pos = df[shape_ok & (df["Price"] > 0)]
hit = pos[pos["Description"].str.contains(words, case=False, na=False)]

print("Rows:", len(hit))
print(hit["Description"].str.strip().value_counts().head(40).to_string())
print()
print(df[df["Description"].str.strip() == "High Resolution Image"].to_string())

Rows: 938
Description
BROWN CHECK CAT DOORSTOP               472
SUNSET CHECK HAMMOCK                   115
PAIR PADDED HANGERS PINK CHECK         100
Next Day Carriage                       80
FRENCH CARRIAGE LANTERN                 63
BLACK BAROQUE CARRIAGE CLOCK            53
MULTICOLOUR CRUSOE CHECK LAMPSHADE      17
WHITE BAROQUE CARRIAGE CLOCK            15
BLUE CHECK BAG W HANDLE 34X20CM         11
BLUE CRUSOE CHECK LAMPSHADE              7
GOTHIC CARRIAGE LANTERN                  4
Dotcomgiftshop Gift Voucher £100.00      1

       Invoice StockCode            Description  Quantity         InvoiceDate  Price  Customer ID         Country  is_duplicate  likely_double_entry
931623  571739     23702  High Resolution Image         1 2011-10-19 10:33:00    3.0      12500.0         Germany         False                False
985841  575910     23702  High Resolution Image         2 2011-11-11 16:05:00    1.5      14836.0  United Kingdom         False                False
997444  576659

In [26]:
miss = df["Description"].isna()
print("Missing description:", miss.sum(),
      "| of those with Price == 0:", (miss & (df["Price"] == 0)).sum())

d = df.dropna(subset=["Description"]).copy()
d["code"] = d["StockCode"].str.strip().str.upper()
d["desc"] = d["Description"].str.strip().str.upper()

n_desc = d.groupby("code")["desc"].nunique()
multi = n_desc[n_desc > 1]
print("\nCodes:", len(n_desc), "| with more than one description:", len(multi))

for c in multi.sort_values(ascending=False).head(8).index:
    print(c, list(d.loc[d["code"] == c, "desc"].value_counts().head(4).index))

Missing description: 4382 | of those with Price == 0: 4382

Codes: 4778 | with more than one description: 1190
20713 ['JUMBO BAG OWLS', 'FOUND', 'WRONGLY MARKED. 23343 IN BOX', 'MISSING']
22423 ['REGENCY CAKESTAND 3 TIER', 'DAMAGED', 'SMASHED', 'DAMAGES']
22734 ['SET OF 6 RIBBONS VINTAGE CHRISTMAS', 'AMAZON', 'CARTON QNTY WAS 216 NOT 144 AS STAT', 'AMAZON ADJUSTMENT']
21181 ['PLEASE ONE PERSON METAL SIGN', 'PLEASE ONE PERSON  METAL SIGN', 'ADJUSTMENT', 'MISSING']
23084 ['RABBIT NIGHT LIGHT', 'TEMP ADJUSTMENT', 'ALLOCATE STOCK FOR DOTCOM ORDERS TA', 'ADD STOCK TO ALLOCATE ONLINE ORDERS']
21830 ['ASSORTED CREEPY CRAWLIES', 'MERCHANT CHANDLER CREDIT ERROR, STO', 'SOLD AS 1', '?']
22719 ['GUMBALL MONOCHROME COAT RACK', 'GUMBALL COATHOOK., BLACK & WHITE', '22467', 'WRONG BARCODE (22467)']
47566B ['TEA TIME PARTY BUNTING', 'INCORRECTLY CREDITED C550456 SEE 47', 'MISSING', 'CORRECT PREVIOUS ADJUSTMENT']


In [27]:
pos = df[(df["Price"] > 0) & df["Description"].notna()].copy()
pos["code"] = pos["StockCode"].str.strip().str.upper()
pos["desc"] = pos["Description"].str.strip().str.upper()

main = pos.groupby("code")["desc"].agg(lambda s: s.value_counts().index[0])
pos["main_desc"] = pos["code"].map(main)
other = pos[pos["desc"] != pos["main_desc"]]

print("Positive-price rows:", len(pos))
print("Rows whose description differs from the code's main one:", len(other))
print(other["desc"].value_counts().head(25).to_string())

Positive-price rows: 1061164
Rows whose description differs from the code's main one: 61238
desc
PACK OF 72 RETRO SPOT CAKE CASES       1466
LUNCH BAG RED SPOTTY                   1275
LUNCH BAG SUKI DESIGN                  1138
SET/20 RED SPOTTY PAPER NAPKINS         967
SET/5 RED SPOTTY LID GLASS BOWLS        952
POPCORN HOLDER                          865
LUNCH BAG PINK RETROSPOT                853
JUMBO BAG PINK WITH WHITE SPOTS         829
RETRO SPOT TEA SET CERAMIC 11 PC        813
RED SPOTTY CHARLOTTE BAG                807
PAPER BUNTING RETRO SPOTS               802
ANTIQUE SILVER T-LIGHT GLASS            719
DOOR MAT UNION FLAG                     682
JUMBO BAG SCANDINAVIAN BLUE PAISLEY     675
PAPER CHAIN KIT RETRO SPOT              603
SET OF 72 RETRO SPOT PAPER  DOILIES     560
JUMBO BAG RED WHITE SPOTTY              551
RED SPOTTY SHOPPER BAG                  523
TOY TIDY PINK RETROSPOT                 523
JUMBO BAG DOILEY PATTERNS               522
LUNCH BAG DOILEY PATTER

In [3]:
code = df["StockCode"].str.strip().str.upper()
non_product = ["POST", "DOT", "C2", "BANK CHARGES", "AMAZONFEE", "CRUK",
               "M", "ADJUST", "ADJUST2", "B", "D", "S", "C3"]

overlap = (df["Price"] == 0) & code.isin(non_product)
print("Zero-price rows with a non-product code:", overlap.sum())
print(code[overlap].value_counts().to_string())

Zero-price rows with a non-product code: 25
StockCode
DOT     7
M       7
POST    7
C2      3
C3      1


In [4]:
cols_show = ["Invoice", "StockCode", "Description", "Quantity",
             "InvoiceDate", "Price", "Customer ID"]
print(df.loc[overlap, cols_show].to_string())

        Invoice StockCode     Description  Quantity          InvoiceDate  Price  Customer ID
5904     489861       DOT  DOTCOM POSTAGE         1  2009-12-02 14:50:00    0.0          NaN
16107    490727         M          Manual         1  2009-12-07 16:38:00    0.0      17231.0
32207    491990        C2             NaN       100  2009-12-15 10:06:00    0.0          NaN
129167   501683       DOT  DOTCOM POSTAGE         1  2010-03-18 16:32:00    0.0          NaN
191175   507450      POST             NaN      1000  2010-05-10 10:01:00    0.0          NaN
270781   515611       DOT  DOTCOM POSTAGE         1  2010-07-13 15:52:00    0.0          NaN
271469   515757       DOT             NaN       500  2010-07-14 13:33:00    0.0          NaN
302041   518785        C3             NaN     -4999  2010-08-11 16:55:00    0.0          NaN
430036   530428        C2             NaN       200  2010-11-03 10:29:00    0.0          NaN
431450   530609      POST             NaN       100  2010-11-03 15:25:

## Decision: classifying rows

**Why**
A row in this file is an invoice line, but not every line is a sale. We found postage, fees, vouchers, test entries, stock write-offs and accounting adjustments mixed in with products. Counting them as sales would distort totals and "top product" rankings.

**Two columns (no rows are deleted)**
- `is_cancellation`: the invoice number starts with "C".
- `row_type`: the first matching rule below.

| Priority | row_type | Rule |
|---|---|---|
| 1 | adjustment | Price < 0 (5 rows, all "Adjust bad debt") |
| 2 | stock_adjustment | Price == 0 (6,202 rows: write-offs, found items, notes; 4,382 have no description) |
| 3 | shipping | Code POST, DOT, C2, or description "Next Day Carriage" |
| 4 | fee | Code BANK CHARGES, AMAZONFEE, CRUK |
| 5 | adjustment | Code M, ADJUST, ADJUST2, B |
| 6 | discount, sample | Code D, S |
| 7 | voucher | Code starting with GIFT, or description starting with "Dotcomgiftshop Gift Voucher" |
| 8 | test | Code starting with TEST |
| 9 | other | Code C3 (no description) |
| 10 | product | Everything else |

**Why description alone is not enough**
- Staff notes and real names share product codes (for example code 20713 has "JUMBO BAG OWLS", "FOUND" and "MISSING").
- Words need care: "carriage" is shipping in "Next Day Carriage" but part of the name in "FRENCH CARRIAGE LANTERN". We match exact descriptions, not words.

**Known uncertainties**
- Codes starting with DCGS, plus PADS and SP1002, are treated as products. Several have blank descriptions.
- "High Resolution Image" (3 rows with a customer and a positive price) is kept as a product.
- Code M ("Manual") is treated as an adjustment, but one row has a price of £373.57 and may be a real charge.
- 71 zero-price rows have a customer ID and real product names. They may be free items, and are classed as stock_adjustment by the price rule.
- The rule list comes from reading the data and may miss rare cases.

In [5]:
import numpy as np

code = df["StockCode"].str.strip().str.upper()
desc = df["Description"].str.strip()

df["is_cancellation"] = df["Invoice"].str.startswith("C")

conditions = [
    df["Price"] < 0,
    df["Price"] == 0,
    code.isin(["POST", "DOT", "C2"]) | (desc == "Next Day Carriage"),
    code.isin(["BANK CHARGES", "AMAZONFEE", "CRUK"]),
    code.isin(["M", "ADJUST", "ADJUST2", "B"]),
    code == "D",
    code == "S",
    code.str.startswith("GIFT") | desc.str.startswith("Dotcomgiftshop Gift Voucher", na=False),
    code.str.startswith("TEST"),
    code == "C3",
]
labels = ["adjustment", "zero_price", "shipping", "fee", "adjustment",
          "discount", "sample", "voucher", "test", "other"]

df["row_type"] = np.select(conditions, labels, default="product")

print(df["row_type"].value_counts())
print("\nTotal rows:", len(df))
print("Cancellation rows:", df["is_cancellation"].sum())

row_type
product       1055226
zero_price       6202
shipping         3913
adjustment       1495
discount          177
fee               161
sample            104
voucher            79
test               14
Name: count, dtype: int64

Total rows: 1067371
Cancellation rows: 19494


In [6]:
code = df["StockCode"].str.strip().str.upper()
is_v = code.str.startswith("GIFT") | df["Description"].str.strip().str.startswith("Dotcomgiftshop Gift Voucher", na=False)
is_t = code.str.startswith("TEST")

print("Voucher-type rows:", is_v.sum(), "| zero price:", (is_v & (df["Price"] == 0)).sum())
print("Test-type rows:   ", is_t.sum(), "| zero price:", (is_t & (df["Price"] == 0)).sum())

Voucher-type rows: 102 | zero price: 23
Test-type rows:    17 | zero price: 3


**Result (1,067,371 rows)**

| row_type | Rows |
|---|---|
| product | 1,055,226 |
| zero_price | 6,202 |
| shipping | 3,913 |
| adjustment | 1,495 |
| discount | 177 |
| fee | 161 |
| sample | 104 |
| voucher | 79 |
| test | 14 |

`is_cancellation` marks 19,494 rows. The `other` rule (code C3) never fires, because that row has a zero price and rule 2 takes it first.

In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

cols = ["Invoice", "StockCode", "Description", "Quantity",
        "InvoiceDate", "Price", "Customer ID", "Country"]

invoice_start = df.groupby("Invoice")["InvoiceDate"].transform("min")
event_rows = (invoice_start >= "2010-12-01") & (invoice_start < "2010-12-10")

df["is_duplicate"] = df.duplicated(keep="first")

group_size = df.groupby(cols, dropna=False)["Invoice"].transform("size")
position = df.groupby(cols, dropna=False).cumcount()
df["likely_double_entry"] = event_rows & (position >= group_size / 2)

print("is_duplicate:", df["is_duplicate"].sum())
print("Rows in event invoices:", event_rows.sum())
print("likely_double_entry:", df["likely_double_entry"].sum())

is_duplicate: 34335
Rows in event invoices: 45046
likely_double_entry: 22523


In [9]:
df["line_value"] = df["Quantity"] * df["Price"]

prod = df["row_type"] == "product"
keep = ~df["likely_double_entry"]

def total(mask):
    return round(df.loc[mask, "line_value"].sum(), 2)

print("Product rows, all:               ", total(prod))
print("  sales (not cancellations):     ", total(prod & ~df["is_cancellation"]))
print("  cancellations:                 ", total(prod & df["is_cancellation"]))
print("Product rows, no double entries: ", total(prod & keep))
print("Double-entry rows (product):     ", total(prod & df["likely_double_entry"]))

Product rows, all:                19382356.7
  sales (not cancellations):      20108897.69
  cancellations:                  -726540.99
Product rows, no double entries:  18979993.19
Double-entry rows (product):      402363.51


In [10]:
flag_nonprod = df["likely_double_entry"] & (df["row_type"] != "product")

print(df.loc[flag_nonprod].groupby("row_type")["line_value"]
        .agg(["count", "sum"]).round(2))
print("\nTotal rows:", flag_nonprod.sum())
print("Total value:", round(df.loc[flag_nonprod, "line_value"].sum(), 2))

            count       sum
row_type                   
adjustment     23    154.30
discount        4   -604.91
fee             9 -39323.46
sample          2    -64.95
shipping       62  14963.96
zero_price    178      0.00

Total rows: 278
Total value: -24875.06


## Decision: what counts as a sale

**Sales analysis rows** = rows with `row_type == "product"` that are not `likely_double_entry`.

- Cancellations stay in the set as negative lines, so totals are *net* of cancellations. They are not matched to the original sale.
- Shipping, fees, discounts, vouchers, samples, adjustments, test and zero-price rows are excluded from sales and can be reported separately.
- Values are quantity × price in pounds sterling. This is line value, not profit; the data has no costs.

**Effect of the double-entry decision**
- Including likely double entries: net product value £19,382,356.70.
- Excluding them: £18,979,993.19, which is 2.08% lower.
- 22,523 rows are flagged: 22,245 product rows and 278 non-product rows (postage, fees and others on the same doubled invoices).

**Limitations**
- Cancellations cannot be linked to their original sale.
- 23% of rows have no Customer ID, so customer-level results cover only part of the data.

In [11]:
df["is_sale"] = (df["row_type"] == "product") & ~df["likely_double_entry"]

s = df[df["is_sale"]]
print("Analysis rows:", len(s))
print("  of which cancellations:", s["is_cancellation"].sum())
print("  net value:", round(s["line_value"].sum(), 2))

Analysis rows: 1032981
  of which cancellations: 17973
  net value: 18979993.19


In [12]:
ids = df["Customer ID"].dropna()
print("All IDs whole numbers:", (ids % 1 == 0).all())

df["customer_id"] = df["Customer ID"].astype("Int64").astype("string")
df["has_customer"] = df["Customer ID"].notna()

s = df[df["is_sale"]]
print("Analysis rows without customer:", (~s["has_customer"]).sum(),
      "of", len(s))
print("Value without customer:",
      round(s.loc[~s["has_customer"], "line_value"].sum(), 2),
      "of", round(s["line_value"].sum(), 2))
print("Distinct customers in analysis rows:", s["customer_id"].nunique())

All IDs whole numbers: True
Analysis rows without customer: 227213 of 1032981
Value without customer: 2569283.44 of 18979993.19
Distinct customers in analysis rows: 5875


## Decision: customers

- `Customer ID` was stored as decimals (for example 13085.0) only because some values are missing. All IDs are whole numbers, so `customer_id` converts them to text. The original column is kept.
- `has_customer` marks rows with an ID.
- 22.0% of sales-analysis rows have no customer ID (13.5% of net value). They stay in total sales but are excluded from customer analysis, so customer results cover about 86.5% of value.

In [13]:
s = df[df["is_sale"] & ~df["is_cancellation"]]

print(s[["Quantity", "Price", "line_value"]].describe().round(2))
print()
top = s.nlargest(5, "line_value")
print(top[["Invoice", "StockCode", "Description", "Quantity",
           "Price", "line_value", "Customer ID"]].to_string())

         Quantity       Price  line_value
count  1015008.00  1015008.00  1015008.00
mean        11.06        3.33       19.41
std        128.01        4.73      198.84
min          1.00        0.00        0.00
25%          1.00        1.25        3.90
50%          3.00        2.10        9.96
75%         12.00        4.13       17.70
max      80995.00     1157.15   168469.60

        Invoice StockCode                          Description  Quantity   Price  line_value  Customer ID
1065882  581483     23843          PAPER CRAFT , LITTLE BIRDIE     80995    2.08    168469.6      16446.0
587080   541431     23166       MEDIUM CERAMIC TOP STORAGE JAR     74215    1.04     77183.6      12346.0
748132   556444     22502       PICNIC BASKET WICKER 60 PIECES        60  649.50     38970.0      15098.0
432176   530715     84347  ROTATING SILVER ANGELS T-LIGHT HLDR      9360    1.69     15818.4      15838.0
228042   511465    15044A                  PINK PAPER PARASOL       3500    2.55      8925.

In [14]:
big = df[(df["Quantity"].abs() >= 74215) & df["is_sale"]]
print(big[["Invoice", "StockCode", "Quantity", "Price",
           "line_value", "Customer ID", "InvoiceDate"]].to_string())

tiny = df[df["is_sale"] & (df["Price"] < 0.01)]
print("\nSale rows with price below 0.01:", len(tiny))
print(tiny[["Invoice", "StockCode", "Description", "Quantity", "Price"]]
      .head(10).to_string())

         Invoice StockCode  Quantity  Price  line_value  Customer ID         InvoiceDate
587080    541431     23166     74215   1.04     77183.6      12346.0 2011-01-18 10:01:00
587085   C541433     23166    -74215   1.04    -77183.6      12346.0 2011-01-18 10:17:00
1065882   581483     23843     80995   2.08    168469.6      16446.0 2011-12-09 09:15:00
1065883  C581484     23843    -80995   2.08   -168469.6      16446.0 2011-12-09 09:27:00

Sale rows with price below 0.01: 17
       Invoice StockCode                 Description  Quantity  Price
62299   494914      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
74731   496222      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
77702   496473      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
79794   496643      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
90798   497935      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
97716   498562      PADS  PADS TO MATCH ALL CUSHIONS         1  0.001
101718  499056      PADS  PA

## Check: extreme values
- The two largest sales (74,215 and 80,995 units) were each cancelled minutes later by the same customer, so they net to £0. This is one reason cancellations stay in the analysis set; without them net sales would be about £245,653 (1.3%) too high.
- 17 sale rows have a price of £0.001 (code PADS, quantity 1). Their combined value is negligible, so they are kept.
- Customer rankings must use net value, otherwise customers with reversed orders rank too high.

In [15]:
c = df[df["is_cancellation"]]
print("Cancellation rows:", len(c))
print("In analysis set:", c["is_sale"].sum())
print()
print(c.loc[~c["is_sale"]].groupby(["row_type", "likely_double_entry"]).size())

Cancellation rows: 19494
In analysis set: 17973

row_type    likely_double_entry
adjustment  False                  566
            True                     3
discount    False                  168
            True                     4
fee         False                  115
            True                     7
product     True                   312
sample      False                   99
            True                     2
shipping    False                  239
            True                     1
test        False                    4
voucher     False                    1
dtype: int64


## Check: cancellations outside the analysis set
- 17,973 of 19,494 cancellation rows are in the analysis set. The other 1,521 are 1,209 non-product lines (postage, fees, adjustments and similar, excluded together with the lines they reverse) and 312 product lines that are the second copy of a doubled cancellation invoice from 1-9 December 2010.
- Cancellations are not matched to specific sales, so net sales are a total, not a per-order figure.

In [16]:
print(df["Country"].value_counts().to_string())

cc = df.dropna(subset=["customer_id"]).groupby("customer_id")["Country"].nunique()
print("\nCustomers with more than one country:", (cc > 1).sum(), "of", len(cc))

Country
United Kingdom          981330
EIRE                     17866
Germany                  17624
France                   14330
Netherlands               5140
Spain                     3811
Switzerland               3189
Belgium                   3123
Portugal                  2620
Australia                 1913
Channel Islands           1664
Italy                     1534
Norway                    1455
Sweden                    1364
Cyprus                    1176
Finland                   1049
Austria                    938
Denmark                    817
Unspecified                756
Greece                     663
Japan                      582
Poland                     535
USA                        535
United Arab Emirates       500
Israel                     371
Hong Kong                  364
Singapore                  346
Malta                      299
Iceland                    253
Canada                     228
Lithuania                  189
RSA                        169


In [17]:
ic = df.groupby("Invoice")["Country"].nunique()
print("Invoices with more than one country:", (ic > 1).sum(), "of", len(ic))

multi = cc[cc > 1].index
x = df[df["customer_id"].isin(multi)]
print()
print(x.groupby("customer_id")["Country"]
        .agg(lambda s: ", ".join(sorted(s.unique()))).to_string())

Invoices with more than one country: 0 of 53628

customer_id
12370           Austria, Cyprus
12394          Belgium, Denmark
12413             France, Spain
12417            Belgium, Spain
12422    Australia, Switzerland
12423          Belgium, Denmark
12429          Austria, Denmark
12431        Australia, Belgium
12449          Belgium, Denmark
12455             Cyprus, Spain
12457       Cyprus, Switzerland
12652           France, Germany
12745      EIRE, United Kingdom


## Decision: country
- Every invoice has exactly one country (0 of 53,628 have more than one), so country is stored per invoice.
- 13 of 5,942 customers appear with two countries, so country is not stored per customer.
- 43 country values include non-standard ones: Unspecified (756 rows), European Community (61), West Indies (54), Channel Islands (1,664) and inconsistent names (EIRE, RSA, USA, Korea). The original column is kept. Name standardisation, if any, will be a separate column.
- 91.9% of rows are from the United Kingdom, so country charts will show the UK separately.

In [18]:
g = df.groupby("Invoice")
print("Invoices with more than one customer value:",
      (g["customer_id"].nunique(dropna=False) > 1).sum())
print("Invoices with more than one timestamp:",
      (g["InvoiceDate"].nunique() > 1).sum())

Invoices with more than one customer value: 0
Invoices with more than one timestamp: 83


In [19]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
g = df.groupby("Invoice")["InvoiceDate"]

spread = g.max() - g.min()
multi = spread[spread > pd.Timedelta(0)]

print("Invoices:", len(multi))
print(multi.describe())
print()
print(multi.sort_values(ascending=False).head(10))

diff_day = g.min().dt.normalize() != g.max().dt.normalize()
print("\nSpanning more than one calendar day:", diff_day[multi.index].sum())

Invoices: 83
count                           83
mean     0 days 00:01:08.674698795
std      0 days 00:00:56.325947846
min                0 days 00:01:00
25%                0 days 00:01:00
50%                0 days 00:01:00
75%                0 days 00:01:00
max                0 days 00:09:00
Name: InvoiceDate, dtype: object

Invoice
529368   0 days 00:09:00
520878   0 days 00:04:00
521901   0 days 00:02:00
547690   0 days 00:01:00
546986   0 days 00:01:00
546388   0 days 00:01:00
545713   0 days 00:01:00
545460   0 days 00:01:00
544926   0 days 00:01:00
544667   0 days 00:01:00
Name: InvoiceDate, dtype: timedelta64[ns]

Spanning more than one calendar day: 0


## Decision: invoice-level fields
- Each invoice has exactly one customer value and one country.
- 83 of 53,628 invoices have lines with different timestamps, but the spread is at most 9 minutes (median 1 minute) and none crosses a calendar day. The invoice table stores the earliest timestamp.
- The database will have three tables: `invoices` (invoice number, date, customer, country), `products` (stock code, one chosen description) and `invoice_lines` (invoice number, stock code, quantity, price).

In [20]:
s = df[df["is_sale"] & ~df["is_cancellation"]]
code = s["StockCode"].str.strip().str.upper()

n_price = s.groupby(code)["Price"].nunique()
print("Codes:", len(n_price), "| with more than one price:", (n_price > 1).sum())
print(n_price.describe())

top = n_price.sort_values(ascending=False).head(5).index
sub = s[code.isin(top)]
print(sub.groupby(code[code.isin(top)])["Price"].agg(["min", "max", "nunique"]))

Codes: 4723 | with more than one price: 4248
count    4723.000000
mean        4.550286
std         2.504414
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        21.000000
Name: Price, dtype: float64
           min    max  nunique
StockCode                     
20685      4.0  16.98       21
21955      4.0  16.98       18
48138      4.0  16.98       18
48187      4.0  16.98       18
48188      4.0  16.98       18


## Decision: price
- 4,248 of 4,723 product codes (90%) have more than one price in the sales rows (median 4 prices per code, maximum 21). Price is therefore stored on `invoice_lines`, not on `products`.
- Per-product "average price" is avoided. Where a unit price is needed, we use line value divided by quantity for a stated period.
- The reason for the price differences is unknown (possibly price changes over time or customer tiers); the data does not say.

In [21]:
s = df[df["is_sale"]]
pairs = pd.DataFrame({
    "code": s["StockCode"].str.strip().str.upper(),
    "desc": s["Description"].str.strip().str.upper(),
})

counts = pairs.groupby(["code", "desc"]).size().rename("n").reset_index()
best = counts.groupby("code")["n"].transform("max")
top = counts[counts["n"] == best]

print("Codes:", counts["code"].nunique())
print("With more than one description:", (counts.groupby("code").size() > 1).sum())
print("Tied for most common:", (top.groupby("code").size() > 1).sum())

tied = top.groupby("code").filter(lambda g: len(g) > 1)
print(tied.head(10).to_string())

Codes: 4737
With more than one description: 599
Tied for most common: 3
        code                               desc   n
752    21291  SMALL POLKADOT CHOCOLATE GIFT BAG  80
753    21291    SMALL SPOTTY CHOCOLATE GIFT BAG  80
4099  84206B        CAT W SUNGLASSES BLANK CARD   1
4100  84206B     CAT WITH SUNGLASSES BLANK CARD   1
4639  84984C              ORIENTAL BLUE C/COVER   8
4640  84984C        ORIENTAL BLUE CUSHION COVER   8


## Decision: one description per product code
- In the sales rows, 599 of 4,737 codes have more than one description (mostly spelling variants or renamed products).
- `products` stores the most common description (cleaned: spaces stripped, capitals). Ties (3 codes) go to the first description alphabetically, so the table can be rebuilt identically.
- The chosen names are a convenience, not a fact about the product.

In [22]:
ordered = counts.sort_values(["code", "n", "desc"], ascending=[True, False, True])
products = (ordered.drop_duplicates("code", keep="first")[["code", "desc"]]
                   .rename(columns={"code": "stock_code", "desc": "description"}))

print("Rows:", len(products), "| unique codes:", products["stock_code"].is_unique)
print(products[products["stock_code"].isin(["21291", "84206B", "84984C"])].to_string())

Rows: 4737 | unique codes: True
     stock_code                        description
752       21291  SMALL POLKADOT CHOCOLATE GIFT BAG
4099     84206B        CAT W SUNGLASSES BLANK CARD
4639     84984C              ORIENTAL BLUE C/COVER


In [23]:
all_codes = df["StockCode"].str.strip().str.upper()
missing = ~all_codes.isin(products["stock_code"])

print("Rows whose code is not in products:", missing.sum(), "of", len(df))
print("Distinct codes not in products:", all_codes[missing].nunique())
print()
print(df.loc[missing, "row_type"].value_counts())

Rows whose code is not in products: 6382 of 1067371
Distinct codes not in products: 394

row_type
shipping      3913
adjustment    1495
zero_price     439
discount       177
fee            161
sample         104
voucher         79
test            14
Name: count, dtype: int64


In [24]:
x = df.loc[missing, ["StockCode", "Description"]].copy()
x["code"] = x["StockCode"].str.strip().str.upper()
x["desc"] = x["Description"].str.strip().str.upper()

xc = (x.dropna(subset=["desc"]).groupby(["code", "desc"])
        .size().rename("n").reset_index())
xo = xc.sort_values(["code", "n", "desc"], ascending=[True, False, True])
extra = (xo.drop_duplicates("code")[["code", "desc"]]
           .rename(columns={"code": "stock_code", "desc": "description"}))

nodesc = pd.DataFrame({"stock_code": sorted(set(x["code"]) - set(extra["stock_code"]))})
nodesc["description"] = pd.NA

base = products.assign(appears_in_sales=True)
more = pd.concat([extra, nodesc], ignore_index=True).assign(appears_in_sales=False)
products_all = pd.concat([base, more], ignore_index=True)

print("Rows:", len(products_all), "| unique codes:", products_all["stock_code"].is_unique)
print("Codes without any description:", products_all["description"].isna().sum())
print("Every row's code covered:", all_codes.isin(products_all["stock_code"]).all())

Rows: 5131 | unique codes: True
Codes without any description: 353
Every row's code covered: True


## Decision: what goes into the database
- All 1,067,371 rows are loaded, with `row_type`, `is_cancellation`, `is_duplicate`, `likely_double_entry` and `is_sale` as columns. Nothing is deleted, and SQL filters on `is_sale`.
- `products` has 5,131 codes. 4,737 appear in sales rows (`appears_in_sales = 1`). The other 394 appear only on non-sale rows, and 353 of them have no description.

In [25]:
inv = (df.groupby("Invoice")
         .agg(invoice_date=("InvoiceDate", "min"),
              customer_id=("customer_id", "first"),
              country=("Country", "first"),
              is_cancellation=("is_cancellation", "first"))
         .reset_index()
         .rename(columns={"Invoice": "invoice_no"}))

lines = pd.DataFrame({
    "line_id": df.index + 1,
    "invoice_no": df["Invoice"],
    "stock_code": df["StockCode"].str.strip().str.upper(),
    "quantity": df["Quantity"],
    "price": df["Price"],
    "row_type": df["row_type"],
    "is_duplicate": df["is_duplicate"],
    "likely_double_entry": df["likely_double_entry"],
    "is_sale": df["is_sale"],
})

print("invoices:", len(inv), "| unique:", inv["invoice_no"].is_unique)
print("lines:", len(lines), "| unique ids:", lines["line_id"].is_unique)
print("lines with unknown invoice:", (~lines["invoice_no"].isin(inv["invoice_no"])).sum())
print("lines with unknown product:", (~lines["stock_code"].isin(products_all["stock_code"])).sum())

sales = lines[lines["is_sale"]]
print("net sales value:", round((sales["quantity"] * sales["price"]).sum(), 2))

invoices: 53628 | unique: True
lines: 1067371 | unique ids: True
lines with unknown invoice: 0
lines with unknown product: 0
net sales value: 18979993.19


In [26]:
import sqlite3
from pathlib import Path

db_path = Path("../data/processed/retail.db")
db_path.parent.mkdir(parents=True, exist_ok=True)
if db_path.exists():
    db_path.unlink()

schema = """
CREATE TABLE invoices (
    invoice_no      TEXT PRIMARY KEY,
    invoice_date    TEXT NOT NULL,
    customer_id     TEXT,
    country         TEXT NOT NULL,
    is_cancellation INTEGER NOT NULL
);
CREATE TABLE products (
    stock_code       TEXT PRIMARY KEY,
    description      TEXT,
    appears_in_sales INTEGER NOT NULL
);
CREATE TABLE invoice_lines (
    line_id             INTEGER PRIMARY KEY,
    invoice_no          TEXT NOT NULL REFERENCES invoices(invoice_no),
    stock_code          TEXT NOT NULL REFERENCES products(stock_code),
    quantity            INTEGER NOT NULL,
    price               REAL NOT NULL,
    row_type            TEXT NOT NULL,
    is_duplicate        INTEGER NOT NULL,
    likely_double_entry INTEGER NOT NULL,
    is_sale             INTEGER NOT NULL
);
"""

con = sqlite3.connect(db_path)
con.executescript(schema)
print(con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
con.close()

[('invoices',), ('products',), ('invoice_lines',)]


## Database design (SQLite)
- Three tables: `invoices` (one row per invoice), `products` (one row per stock code) and `invoice_lines` (one row per line, linking the other two).
- Price lives on `invoice_lines`, because 90% of codes have more than one price.
- All rows are loaded, with flag columns. Analyses filter on `is_sale = 1`.
- Dates are stored as text in ISO format, flags as 0/1 integers.

In [27]:
inv_db = inv.copy()
inv_db["invoice_date"] = inv_db["invoice_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
inv_db["is_cancellation"] = inv_db["is_cancellation"].astype(int)
inv_db["customer_id"] = inv_db["customer_id"].astype(object).where(inv_db["customer_id"].notna(), None)

prod_db = products_all.copy()
prod_db["appears_in_sales"] = prod_db["appears_in_sales"].astype(int)
prod_db["description"] = prod_db["description"].astype(object).where(prod_db["description"].notna(), None)

lines_db = lines.copy()
for c in ["is_duplicate", "likely_double_entry", "is_sale"]:
    lines_db[c] = lines_db[c].astype(int)

con = sqlite3.connect(db_path)
con.execute("PRAGMA foreign_keys = ON")

inv_db.to_sql("invoices", con, if_exists="append", index=False, chunksize=50000)
prod_db.to_sql("products", con, if_exists="append", index=False, chunksize=50000)
lines_db.to_sql("invoice_lines", con, if_exists="append", index=False, chunksize=50000)
con.commit()

for t in ["invoices", "products", "invoice_lines"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])
print("Foreign key violations:", len(con.execute("PRAGMA foreign_key_check").fetchall()))
print("Net sales (SQL):", con.execute(
    "SELECT ROUND(SUM(quantity * price), 2) FROM invoice_lines WHERE is_sale = 1"
).fetchone()[0])
con.close()

invoices 53628
products 5131
invoice_lines 1067371
Foreign key violations: 0
Net sales (SQL): 18979993.19


---------------------------------------------------------------------------------------------------------------------------------------

## SQL analysis

In [28]:
con = sqlite3.connect(db_path)

query = """
SELECT strftime('%Y-%m', i.invoice_date) AS month,
       ROUND(SUM(l.quantity * l.price), 2) AS net_sales,
       COUNT(DISTINCT CASE WHEN i.is_cancellation = 0 THEN i.invoice_no END) AS invoices
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice_no = l.invoice_no
WHERE l.is_sale = 1
GROUP BY month
ORDER BY month;
"""

monthly = pd.read_sql_query(query, con)
con.close()

print(monthly.to_string())
print("\nMonths:", len(monthly))
print("Sum of net_sales:", round(monthly["net_sales"].sum(), 2))

      month   net_sales  invoices
0   2009-12   781541.69      1666
1   2010-01   606385.18      1049
2   2010-02   527160.30      1189
3   2010-03   754676.65      1647
4   2010-04   638968.56      1435
5   2010-05   608351.25      1484
6   2010-06   672935.41      1612
7   2010-07   617764.26      1507
8   2010-08   663642.07      1402
9   2010-09   844100.37      1789
10  2010-10  1072467.65      2244
11  2010-11  1400559.24      2719
12  2010-12   760323.91      1550
13  2011-01   580406.89      1081
14  2011-02   500510.33      1093
15  2011-03   680582.31      1440
16  2011-04   482901.21      1235
17  2011-05   732229.18      1668
18  2011-06   724841.03      1525
19  2011-07   677805.20      1452
20  2011-08   702365.65      1339
21  2011-09  1013340.68      1818
22  2011-10  1062557.66      2005
23  2011-11  1432359.99      2751
24  2011-12   441216.51       816

Months: 25
Sum of net_sales: 18979993.18


## SQL analysis 1: monthly net sales
- Joins invoice_lines to invoices to use the invoice date, filters on is_sale = 1, and groups by month.
- Net sales include cancellations as negative lines. The invoice count excludes cancellation invoices.
- December 2011 is a partial month (data ends on 9 December), so it is not comparable with other months.

## Result: monthly net sales
- Net sales peak in September to November in both years (November: £1.40M in 2010, £1.43M in 2011).
- January to November 2011 is about 2.2% above 2010 over the same months, but individual months vary a lot (April -24%, May +20%), so no growth claim is made.
- December 2011 is a partial month. December 2010 excludes 22,523 likely double-entry rows; with them it would be about £0.4M higher.
- Two years of data cannot separate seasonality from one-off events.

In [29]:
con = sqlite3.connect(db_path)

query = """
SELECT p.stock_code,
       p.description,
       SUM(l.quantity) AS units,
       ROUND(SUM(l.quantity * l.price), 2) AS net_sales,
       ROUND(100.0 * SUM(l.quantity * l.price) /
             (SELECT SUM(quantity * price) FROM invoice_lines WHERE is_sale = 1), 2) AS pct_of_total
FROM invoice_lines AS l
JOIN products AS p ON p.stock_code = l.stock_code
WHERE l.is_sale = 1
GROUP BY p.stock_code, p.description
ORDER BY net_sales DESC
LIMIT 10;
"""

top10 = pd.read_sql_query(query, con)
con.close()

print(top10.to_string())
print("\nTop 10 share of total (%):", round(top10["pct_of_total"].sum(), 2))

  stock_code                          description  units  net_sales  pct_of_total
0      22423             REGENCY CAKESTAND 3 TIER  25067  314513.47          1.66
1     85123A   WHITE HANGING HEART T-LIGHT HOLDER  91202  252122.83          1.33
2     85099B              JUMBO BAG RED RETROSPOT  96137  180830.85          0.95
3      47566                        PARTY BUNTING  27931  147157.43          0.78
4      84879        ASSORTED COLOUR BIRD ORNAMENT  79782  128907.01          0.68
5      22086       PAPER CHAIN KIT 50'S CHRISTMAS  34536  116422.49          0.61
6      79321                        CHILLI LIGHTS  15773   80237.48          0.42
7      22197                 SMALL POPCORN HOLDER  87733   78935.92          0.42
8      22386              JUMBO BAG PINK POLKADOT  38985   75518.22          0.40
9      84347  ROTATING SILVER ANGELS T-LIGHT HLDR  22052   71031.15          0.37

Top 10 share of total (%): 7.62


## Result: top products (net sales, analysis rows)
- Sales are spread across many products: the top product (REGENCY CAKESTAND 3 TIER) is 1.66% of net sales and the top 10 together are 7.62%.
- Rankings differ by value and by units (the most units: JUMBO BAG RED RETROSPOT, 96,137).
- Values are net of cancellations, exclude likely double entries, and are line value rather than profit.
- The dataset has no product category, so no category analysis is made.

In [30]:
con = sqlite3.connect(db_path)

query = """
SELECT i.country,
       ROUND(SUM(l.quantity * l.price), 2) AS net_sales,
       COUNT(DISTINCT CASE WHEN i.is_cancellation = 0 THEN i.invoice_no END) AS invoices,
       ROUND(100.0 * SUM(l.quantity * l.price) /
             (SELECT SUM(quantity * price) FROM invoice_lines WHERE is_sale = 1), 2) AS pct_of_total
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice_no = l.invoice_no
WHERE l.is_sale = 1
GROUP BY i.country
ORDER BY net_sales DESC;
"""

by_country = pd.read_sql_query(query, con)
con.close()

print(by_country.head(10).to_string())
print("\nCountries:", len(by_country))
print("Sum of net_sales:", round(by_country["net_sales"].sum(), 2))

          country    net_sales  invoices  pct_of_total
0  United Kingdom  16219812.97     36184         85.46
1            EIRE    603407.10       581          3.18
2     Netherlands    546097.08       216          2.88
3         Germany    375132.18       753          1.98
4          France    293626.34       598          1.55
5       Australia    166349.65        89          0.88
6     Switzerland     92940.36        85          0.49
7           Spain     84862.88       144          0.45
8          Sweden     84379.07        99          0.44
9         Denmark     63355.59        42          0.33

Countries: 43
Sum of net_sales: 18979993.19


## Result: net sales by country
- The United Kingdom accounts for 85.46% of net sales (£16.22M); the other 42 countries share 14.54%.
- The largest non-UK markets are EIRE (3.18%), the Netherlands (2.88%) and Germany (1.98%).
- The Netherlands has few invoices (216) but high net sales per invoice (about £2,500, against about £450 in the UK), so its total may depend on a few large orders. This was not investigated.
- Country is stored per invoice. 13 customers appear in two countries, so country is not a customer attribute.
- Values are net of cancellations and exclude likely double entries.

In [31]:
con = sqlite3.connect(db_path)

query = """
SELECT i.customer_id,
       COUNT(DISTINCT CASE WHEN i.is_cancellation = 0 THEN i.invoice_no END) AS invoices,
       ROUND(SUM(l.quantity * l.price), 2) AS net_sales,
       ROUND(100.0 * SUM(l.quantity * l.price) /
             (SELECT SUM(l2.quantity * l2.price)
              FROM invoice_lines AS l2
              JOIN invoices AS i2 ON i2.invoice_no = l2.invoice_no
              WHERE l2.is_sale = 1 AND i2.customer_id IS NOT NULL), 2) AS pct_of_customer_sales
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice_no = l.invoice_no
WHERE l.is_sale = 1 AND i.customer_id IS NOT NULL
GROUP BY i.customer_id
ORDER BY net_sales DESC;
"""

by_customer = pd.read_sql_query(query, con)
con.close()

print(by_customer.head(10).to_string())
print("\nCustomers:", len(by_customer))
print("Sum of net_sales:", round(by_customer["net_sales"].sum(), 2))
print("Customers with net_sales <= 0:", (by_customer["net_sales"] <= 0).sum())

  customer_id  invoices  net_sales  pct_of_customer_sales
0       18102       145  578408.64                   3.52
1       14646       145  523202.74                   3.19
2       14156       144  297961.56                   1.82
3       14911       373  258601.47                   1.58
4       17450        51  233803.91                   1.42
5       13694       143  190354.06                   1.16
6       17511        60  168504.44                   1.03
7       12415        24  143107.02                   0.87
8       16684        55  141530.29                   0.86
9       15061       127  124982.13                   0.76

Customers: 5875
Sum of net_sales: 16410709.73
Customers with net_sales <= 0: 44


## Result: customers (rows with a customer ID only)
- 5,875 customers; their net sales total £16.41M, which is 86.5% of all net sales.
- The top 10 customers account for 16.2% of customer sales (about 14% of all net sales). The top two are 3.5% and 3.2%.
- Net sales per invoice vary widely between large customers (about £700 to £4,000).
- 44 customers have net sales at or below zero. The reason is not yet established.
- Customer figures exclude the 13.5% of net sales with no customer ID.

In [32]:
con = sqlite3.connect(db_path)

query = """
SELECT i.customer_id,
       ROUND(SUM(CASE WHEN i.is_cancellation = 0 THEN l.quantity * l.price ELSE 0 END), 2) AS sales,
       ROUND(SUM(CASE WHEN i.is_cancellation = 1 THEN l.quantity * l.price ELSE 0 END), 2) AS cancellations,
       ROUND(SUM(l.quantity * l.price), 2) AS net,
       MIN(i.invoice_date) AS first_date
FROM invoice_lines AS l
JOIN invoices AS i ON i.invoice_no = l.invoice_no
WHERE l.is_sale = 1 AND i.customer_id IS NOT NULL
GROUP BY i.customer_id
HAVING SUM(l.quantity * l.price) <= 0
ORDER BY net;
"""

neg = pd.read_sql_query(query, con)
con.close()

print("Customers:", len(neg))
print("With no sales at all (sales = 0):", (neg["sales"] == 0).sum())
print(neg.head(10).to_string())
print("\nTotal net of these customers:", round(neg["net"].sum(), 2))

Customers: 44
With no sales at all (sales = 0): 23
  customer_id    sales  cancellations      net           first_date
0       13091  1108.80       -2452.04 -1343.24  2009-12-08 12:25:00
1       14337     0.00        -658.63  -658.63  2009-12-08 14:04:00
2       15767     0.00        -242.70  -242.70  2009-12-04 14:56:00
3       15935   108.04        -308.82  -200.78  2009-12-08 17:23:00
4       17755     0.00         -71.23   -71.23  2010-10-11 15:45:00
5       16853     0.00         -67.50   -67.50  2009-12-08 17:32:00
6       17661     0.00         -54.00   -54.00  2010-03-08 16:28:00
7       14864     0.00         -48.75   -48.75  2009-12-08 17:07:00
8       16512     0.00         -33.90   -33.90  2009-12-11 14:31:00
9       12896     0.00         -29.75   -29.75  2009-12-09 09:28:00

Total net of these customers: -2955.59
